# 34. RAPTOR

Recursively cluster and summarise chunks into a tree; retrieve across all levels.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Flat chunk retrieval can answer details, but not questions about a chapter's gist or the whole document. **RAPTOR** builds a *tree*: cluster similar chunks, summarise each cluster, then cluster and summarise the summaries, repeating until a root. Retrieval searches all levels at once (collapsed tree), so detail questions hit leaves and broad questions hit summaries.

## How it works

1. Embed leaf chunks
2. Cluster them (KMeans here; GMM in the paper)
3. Summarise each cluster with the LLM -> next level
4. Repeat until few nodes remain
5. Index every node from every level together
6. Retrieve across levels

```text
 level 2:            [ S ]                      <- gist of everything
 level 1:     [S1]  [S2]  [S3]                  <- cluster summaries
 level 0: c1 c2 c3 c4 c5 c6 c7 c8 c9 ...         <- original chunks
 retrieval searches ALL levels together
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** `pip install scikit-learn`

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 11:11:23.662000 9890 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


In [2]:
import numpy as np
from sklearn.cluster import KMeans

def build_tree(leaf_chunks, max_levels=3, cluster_size=4):
    levels = [leaf_chunks]
    for _ in range(max_levels):
        current = levels[-1]
        if len(current) <= cluster_size:
            break
        n_clusters = max(2, len(current) // cluster_size)
        labels = KMeans(n_clusters=n_clusters, n_init="auto", random_state=0).fit_predict(embed(current))
        summaries = []
        for c in range(n_clusters):
            members = [t for t, l in zip(current, labels) if l == c]
            summaries.append(llm("Summarise these passages in 3 sentences:\n\n" + "\n\n".join(members)))
        levels.append(summaries)
    return levels

tree = build_tree(chunks)
print([len(level) for level in tree], "nodes per level (leaves first)")

[19, 4] nodes per level (leaves first)


## Collapsed-tree retrieval
Put every node from every level into one index. Detail questions hit leaves; broad questions hit summaries.

In [3]:
nodes, meta = [], []
for depth, level in enumerate(tree):
    nodes += level; meta += [{"level": depth}] * len(level)
tree_store = build_store(nodes, meta)

for q in ["What is the dimension of all-MiniLM-L6-v2 vectors?", "What are the major topics covered in this handbook?"]:
    hits = tree_store.search(q, 3)
    print(q, "-> levels hit:", [h["meta"]["level"] for h in hits])
    print(answer(q, [h["text"] for h in hits])[:300], "\n")

What is the dimension of all-MiniLM-L6-v2 vectors? -> levels hit: [0, 0, 0]


According to [1], the all-MiniLM-L6-v2 model produces 384-dimensional vectors. 

What are the major topics covered in this handbook? -> levels hit: [1, 0, 1]


I don't know. The provided context does not mention any specific handbook or its topics. It appears to be related to natural language processing (NLP) and retrieval-augmented generation (RAG) systems, but it doesn't provide information about the major topics of a handbook. 



## Inspect the tree

In [4]:
for depth, level in enumerate(tree):
    print(f"--- level {depth}: {len(level)} nodes")
    for n in level[:2]: print("   ", n[:140].replace("\n", " "))

--- level 0: 19 nodes
    # The Applied AI Handbook  ## Introduction to RAG  Retrieval-Augmented Generation (RAG) combines a retriever with a language model. Instead 
    has four stages: ingest documents, split them into chunks, index the chunks as embeddings, and at query time retrieve the closest chunks and
--- level 1: 4 nodes
    Here is a 3-sentence summary:  Evaluating RAG (Retrieval-Augmented Generation) systems involves assessing both retrieval and generation comp
    Here is a 3-sentence summary of the passages:  Retrieval-Augmented Generation (RAG) combines a retriever with a language model to search an 


## Which level answers which kind of question?

In [5]:
import pandas as pd
probes = {"detail": ["What is the Reciprocal Rank Fusion constant?", "How many dimensions does all-MiniLM-L6-v2 produce?"],
          "broad": ["What are the main topics of the handbook?", "Summarise the techniques that improve retrieval quality."]}
rows = []
for kind, qs_ in probes.items():
    for q in qs_:
        lv = [h["meta"]["level"] for h in tree_store.search(q, 3)]
        rows.append({"type": kind, "question": q[:55], "levels_hit": lv, "mean_level": sum(lv) / len(lv)})
pd.DataFrame(rows)

,type,question,levels_hit,mean_level
0,detail,What is the Reciprocal Rank Fusion constant?,"[0, 0, 0]",0.000000
1,detail,How many dimensions does all-MiniLM-L6-v2 prod...,"[0, 0, 0]",0.000000
2,broad,What are the main topics of the handbook?,"[0, 1, 0]",0.333333
3,broad,Summarise the techniques that improve retrieva...,"[1, 1, 0]",0.666667


## Compare against flat retrieval on a broad question

In [6]:
broad = "What are the main topics and themes covered in the handbook?"
print("FLAT RAG:\n", rag(broad, store, 3)[0], "\n")
print("RAPTOR:\n", answer(broad, [h["text"] for h in tree_store.search(broad, 3)]))

FLAT RAG:
 I don't know. The provided context does not mention a handbook or its content. It discusses various natural language processing (NLP) techniques, including agents and agentic RAG, knowledge graph stores, and chunking strategies, but it does not provide information about handbooks or their topics. 



RAPTOR:
 I don't know. The context doesn't provide information about the specific content or structure of the handbook, nor does it mention any tools or methods for extracting main topics and themes from text.


In [7]:
compare_pipelines({"flat RAG": lambda q: rag(q, store, 3)[0], "RAPTOR": lambda q: answer(q, [h["text"] for h in tree_store.search(q, 3)])}, n=5)

,keyword_recall,judge_correct,avg_seconds
pipeline,,,
flat RAG,0.81,1.0,0.6
RAPTOR,0.83,0.8,0.7


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Handles both detail and big-picture questions | Expensive build (cluster + summarise) |
| Retrieval is a single standard vector search | Needs rebuilding when documents change |
| Summaries are reusable | Summary errors propagate upwards |

## When to use it

Long documents and books where questions range from facts to overall themes.

## Experiments to try

- Use Gaussian mixture models with soft clustering.
- Try tree-traversal retrieval instead of collapsed tree.
- Limit tree depth to 2.

## Key takeaways

- Recursive summarisation creates abstraction levels.
- Collapsed-tree retrieval is simple and effective.
- Build cost is the trade-off.

---
**Next:** [35. MemoRAG](../35_memorag/memorag.ipynb)